In [1]:
import os
import math
import torch
from torch import nn, optim
from tqdm import tqdm

In [2]:
#choose cpu/gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
#load U-Net
%run U-Net.ipynb
#load Loss Function
%run Loss-Function.ipynb
#load EMA
%run EMA.ipynb
#load evaluation
%run Evaluation.ipynb

In [5]:
#save checkpoint
def save_checkpoint(model, optimizer, step, save_dir):
    #create model dir
    if not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)
    #create check point path
    check_point_path = os.path.join(save_dir, f'step_{step}_checkpoint.pth')
    #create check point status
    check_point = {
        'step':step,
        'model':model.state_dict(),
        'optimizer':optimizer.state_dict()
    }
    #save parameters
    torch.save(check_point, check_point_path)

In [21]:
#train function
def train_loop(model, ema, optimizer, max_steps, loader):
    #train
    loss_list = []
    sigma_log_list = []
    FID_list = []
    model.train()
    step = 0
    
    while step < max_steps:
            #show the pbar
            pbar = tqdm(loader, desc='Train')
            for batch_idx, (x, _) in enumerate(pbar):
                x = x.to(device)
                loss, sigma = EDM_loss(model, x)
                #back propagation
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                #update ema
                ema.update(model)
                optimizer.zero_grad()
                #collect loss value
                loss_list.append(loss.item())
                sigma_log_list.append(math.log(sigma.mean().item()))
                
                step += 1
                #update the bar
                if step % 100 == 0:
                    print(f'Step: {step} | Loss Value: {loss.item():.4f}')
                if step > max_steps:
                    break
                if step % 10000 == 0:
                    save_checkpoint(model, optimizer, step, save_dir='checkpoint')
                #save FID value
                if step % 2000 == 0:
                    #change to ema weights
                    ema.apply_shadow(model)
                    #generate samples
                    generate_sample(model, save_dir='alpha00_samples', num_samples=5000, batch_size=64, image_size=32, use_tqdm=False)
                    #calculate FID
                    real_dir = 'CIFAR10_real_images'
                    fake_dir = 'alpha00_samples'
                    fid_value = compute_fid(real_dir, fake_dir)
                    FID_list.append(fid_value.item())
                    print(f'Step: {step} -> FID: {fid_value}')
                    #restore to original ema params
                    ema.restore(model)
                    
    return loss_list, sigma_log_list, FID_list